In [1]:
pip install apify-client

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.5/85.5 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.1/6.1 MB 51.0 MB/s eta 0:00:00


In [3]:
import time
from datetime import datetime, timedelta
from apify_client import ApifyClient

# ===== CONFIGURATION =====
APIFY_TOKEN = "apify_api_7VvRuShRdfoms1g9qpZ6zha3Xw8oLx2bscxt"
HASHTAGS = ["wedding dresses"]
RESULTS_PER_PAGE = 100
OLD_DAYS_THRESHOLD = 90
WEIGHTS = {
    "likes": 0.25,
    "comments": 0.3,
    "shares": 0.35,
    "views": 0.10
}

# ===== INIT CLIENT =====
client = ApifyClient(APIFY_TOKEN)


def run_tiktok_hashtag_scraper(hashtags, results_per_page=50):
    run_input = {
        "hashtags": hashtags,
        "resultsPerPage": results_per_page,
    }
    print(f"Starting TikTok Hashtag Scraper for: {hashtags} ...")
    run = client.actor("clockworks/tiktok-hashtag-scraper").call(run_input=run_input)

    dataset_id = run.get("defaultDatasetId")
    if not dataset_id:
        raise ValueError("No dataset found for this run.")

    print("Waiting for scraper to finish...")
    time.sleep(20)

    dataset_items = client.dataset(dataset_id).list_items(limit=results_per_page)
    print(f"Total items found: {dataset_items.total}")
    return dataset_items.items


def tag_old_videos(items, days_threshold=90):
    tagged_items = []
    cutoff_date = datetime.now() - timedelta(days=days_threshold)

    for item in items:
        created_time = item.get("createTime") or item.get("createTimeInSeconds")
        if not created_time:
            tagged_items.append((item, "", 0))
            continue

        if isinstance(created_time, (int, float)):
            video_date = datetime.fromtimestamp(created_time)
        elif isinstance(created_time, str):
            try:
                video_date = datetime.fromtimestamp(float(created_time))
            except:
                tagged_items.append((item, "", 0))
                continue
        else:
            tagged_items.append((item, "", 0))
            continue

        label = "🔥 Resurging Trend" if video_date < cutoff_date else ""
        popularity_score = calculate_popularity_score(item)
        tagged_items.append((item, label, popularity_score))

    return tagged_items


def calculate_popularity_score(item):
    likes = item.get("diggCount", 0)
    comments = item.get("commentCount", 0)
    shares = item.get("shareCount", 0)
    views = item.get("playCount", 0)

    score = (
        likes * WEIGHTS["likes"] +
        comments * WEIGHTS["comments"] +
        shares * WEIGHTS["shares"] +
        views * WEIGHTS["views"]
    )
    return score


def print_scraped_data_separated(items_with_labels):
    if not items_with_labels:
        print("No data found!")
        return

    resurging_posts = [x for x in items_with_labels if x[1] == "🔥 Resurging Trend"]
    non_resurging_posts = [x for x in items_with_labels if x[1] != "🔥 Resurging Trend"]

    # Sort each by popularity score
    resurging_sorted = sorted(resurging_posts, key=lambda x: x[2], reverse=True)[:20]
    non_resurging_sorted = sorted(non_resurging_posts, key=lambda x: x[2], reverse=True)[:20]

    def print_list(items, title):
        print(f"\n=== {title} ===")
        for idx, (item, label, score) in enumerate(items, start=1):
            video_url = item.get("webVideoUrl")
            likes = item.get("diggCount", 0)
            shares = item.get("shareCount", 0)
            comments = item.get("commentCount", 0)
            views = item.get("playCount", 0)
            creator = item.get("authorMeta", {}).get("name", "Unknown")
            description = item.get("text", "")
            created_time = item.get("createTime") or item.get("createTimeInSeconds", "")

            print(f"{idx}. Video URL: {video_url}")
            print(f"   Popularity Score: {score:.2f}")
            print(f"   Views: {views}")
            print(f"   Likes: {likes}")
            print(f"   Shares: {shares}")
            print(f"   Comments: {comments}")
            print(f"   Creator: {creator}")
            print(f"   Description: {description}")
            print(f"   Created Time: {created_time}")
            if label:
                print(f"   Label: {label}")
            print("-" * 50)

    print_list(resurging_sorted, "Top 20 Resurging Trend Posts")
    print_list(non_resurging_sorted, "Top 20 Non-Resurging Posts")


if __name__ == "__main__":
    try:
        items = run_tiktok_hashtag_scraper(HASHTAGS, RESULTS_PER_PAGE)
        tagged_items = tag_old_videos(items, OLD_DAYS_THRESHOLD)
        print_scraped_data_separated(tagged_items)
    except Exception as e:
        print(f"Error: {e}")

Starting TikTok Hashtag Scraper for: ['wedding dresses'] ...


[apify.tiktok-hashtag-scraper runId:XWtWDZtbVdEqbngk6] -> Status: RUNNING, Message: 
[apify.tiktok-hashtag-scraper runId:XWtWDZtbVdEqbngk6] -> 2025-10-09T21:40:37.530Z ACTOR: Pulling container image of build seQd0mYq3niYKfC1z from registry.
[apify.tiktok-hashtag-scraper runId:XWtWDZtbVdEqbngk6] -> 2025-10-09T21:40:37.535Z ACTOR: Creating container.
[apify.tiktok-hashtag-scraper runId:XWtWDZtbVdEqbngk6] -> 2025-10-09T21:40:37.580Z ACTOR: Starting container.
[apify.tiktok-hashtag-scraper runId:XWtWDZtbVdEqbngk6] -> 2025-10-09T21:40:37.781Z Will run command: xvfb-run -a -s "-ac -screen 0 1920x1080x24+32 -nolisten tcp" /bin/sh -c ./start_xvfb_and_run_cmd.sh && npm run start:prod --silent
[apify.tiktok-hashtag-scraper runId:XWtWDZtbVdEqbngk6] -> 2025-10-09T21:40:39.715Z INFO  System info {"apifyVersion":"3.4.2","apifyClientVersion":"2.12.6","crawleeVersion":"3.13.9","osType":"Linux","nodeVersion":"v20.19.5"}
[apify.tiktok-hashtag-scraper runId:XWtWDZtbVdEqbngk6] -> Status: RUNNING, Message:

Waiting for scraper to finish...
Total items found: 100

=== Top 20 Resurging Trend Posts ===
1. Video URL: https://www.tiktok.com/@sagittariusxxqueen/video/7354501105675619590
   Popularity Score: 15978525.00
   Views: 122200000
   Likes: 13700000
   Shares: 888300
   Comments: 75400
   Creator: sagittariusxxqueen
   Description: #fyp #dresses #weddingdress #fypシ #dream #gown #iconicdress #fyy #fyyyyyyyyyyyyyyyy #dress #foryou #foryoupage #fashion #dreamdress 
   Created Time: 1712353230
   Label: 🔥 Resurging Trend
--------------------------------------------------
2. Video URL: https://www.tiktok.com/@g.juliet_/video/7407948730562907438
   Popularity Score: 3563358.10
   Views: 26300000
   Likes: 3400000
   Shares: 230600
   Comments: 8827
   Creator: g.juliet_
   Description: Que bonito tu vestido blanco🥹🤍💍

#vestidoblanco #boda #quebonitotuvestidoblanco#wedding#novia#bride#dress #bodamexicana #weddingdress #amandanoviasdress #amandanovias 
   Created Time: 1724797478
   Label: 🔥 Re